In [3]:
"""
Graph 2 — Per-stage processing time: video pipeline modes + podcast generation.
Data source:
  - Table 2 (video stage times) + Table 3 (video end-to-end totals)
  - Table 4 (podcast end-to-end totals: 2 speakers, 15 pages)

Video stage derivations:
  Fixed stages total 69 s (Table 2); narration-only total = 180 s → overhead = 111 s (constant).
  Avatar compositing: 360 − 180 = 180 s.
  XTTS overhead:      660 − 360 = 300 s extra TTS time.

Podcast stage derivations (estimated from implementation + totals):
  PDF Parsing: 3 s (fixed).
  AI Script Gen: ~15 s per 20-min segment (1/2/3 segments for 20/40/60 min).
  TTS Multi-speaker: derived as remainder after fixed costs.
  Audio Mix + Export: ~14 s (fixed).
"""

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import numpy as np

matplotlib.use('Agg')
matplotlib.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.grid': True,
    'axes.grid.axis': 'x',
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'grid.linewidth': 0.6,
})

# ── Video pipeline data (seconds) ────────────────────────────────────────────
VIDEO_STAGES = [
    'System Overhead / Export',
    'Video Finalisation',
    'Avatar Compositing',
    'TTS / Voice Synthesis',
    'PPTX Rendering',
    'AI Content Gen (DeepSeek)',
    'PDF Parsing',
]
VIDEO_DATA = {
    'System Overhead / Export':  [111, 111, 111],
    'Video Finalisation':        [ 15,  15,  15],
    'Avatar Compositing':        [  0, 180, 180],
    'TTS / Voice Synthesis':     [ 30,  30, 330],
    'PPTX Rendering':            [  3,   3,   3],
    'AI Content Gen (DeepSeek)': [ 18,  18,  18],
    'PDF Parsing':               [  3,   3,   3],
}
VIDEO_MODES   = ['Narration Only', 'AI Avatar (Edge TTS)', 'Voice Cloning (XTTS v2)']
VIDEO_TOTALS  = [180, 360, 660]

# ── Podcast data (seconds) ───────────────────────────────────────────────────
# Totals from Table 4: 1:36.72, 3:39.80, 4:07.54
PODCAST_TOTALS_S = [96.72, 219.80, 247.54]
PODCAST_LABELS   = ['Podcast — 20 min', 'Podcast — 40 min', 'Podcast — 60 min']

# Estimated per-stage breakdown derived from totals + implementation knowledge
PODCAST_STAGES = [
    'Audio Mix + Export',
    'TTS Multi-speaker (Edge TTS)',
    'AI Script Gen (DeepSeek)',
    'PDF Parsing',
]
# Segments: 1 DeepSeek call / 20-min segment → 1, 2, 3 calls
_script_gen  = [15, 28, 38]   # ~15 s per API call
_fixed       = 3              # PDF parsing
_audio_mix   = 14             # fixed mixing + EBU R128 + export
_tts         = [
    round(t - _fixed - g - _audio_mix, 2)
    for t, g in zip(PODCAST_TOTALS_S, _script_gen)
]  # remainder goes to TTS
PODCAST_DATA = {
    'Audio Mix + Export':          [_audio_mix] * 3,
    'TTS Multi-speaker (Edge TTS)': _tts,
    'AI Script Gen (DeepSeek)':    _script_gen,
    'PDF Parsing':                 [_fixed] * 3,
}

# ── Colour palettes ──────────────────────────────────────────────────────────
VIDEO_COLORS = {
    'PDF Parsing':               '#B0C4DE',
    'AI Content Gen (DeepSeek)': '#5B9BD5',
    'PPTX Rendering':            '#A8D5BA',
    'TTS / Voice Synthesis':     '#F4845F',
    'Avatar Compositing':        '#9B59B6',
    'Video Finalisation':        '#48C9B0',
    'System Overhead / Export':  '#D5D8DC',
}
PODCAST_COLORS = {
    'PDF Parsing':                 '#B0C4DE',        # shared with video
    'AI Script Gen (DeepSeek)':    '#5B9BD5',        # shared hue with AI Content Gen
    'TTS Multi-speaker (Edge TTS)':'#E8954A',        # warm orange, distinct from video TTS
    'Audio Mix + Export':          '#74B9A0',        # teal-green
}

# ── Build figure ─────────────────────────────────────────────────────────────
n_video   = len(VIDEO_MODES)
n_podcast = len(PODCAST_LABELS)
SPACING   = 1.6   # units between bar centres — prevents label collision
GAP       = 2.0   # extra gap between video and podcast groups

y_video   = np.arange(n_video,   dtype=float) * SPACING
y_podcast = np.arange(n_podcast, dtype=float) * SPACING + n_video * SPACING + GAP

all_y      = np.concatenate([y_video, y_podcast])
all_labels = VIDEO_MODES + PODCAST_LABELS

fig, ax = plt.subplots(figsize=(11, 9.0))
bar_h = 0.60

# ── Draw video bars ──────────────────────────────────────────────────────────
for stage in reversed(VIDEO_STAGES):
    vals    = np.array(VIDEO_DATA[stage], dtype=float)
    bottoms = np.zeros(n_video)
    for prev_stage in reversed(VIDEO_STAGES):
        if prev_stage == stage:
            break
        bottoms += np.array(VIDEO_DATA[prev_stage], dtype=float)

bottoms_v = {s: np.zeros(n_video) for s in VIDEO_STAGES}
cumulative = np.zeros(n_video)
for stage in reversed(VIDEO_STAGES):
    vals = np.array(VIDEO_DATA[stage], dtype=float)
    ax.barh(y_video, vals, left=cumulative, height=bar_h,
            color=VIDEO_COLORS[stage], edgecolor='white', linewidth=0.6, zorder=3)
    for xi, (v, b) in enumerate(zip(vals, cumulative)):
        if v >= 18:
            ax.text(b + v / 2, y_video[xi], f'{v:.0f}s',
                    ha='center', va='center', fontsize=7, color='white', fontweight='bold')
    cumulative += vals

for xi, total in enumerate(VIDEO_TOTALS):
    mins = total // 60
    secs = total % 60
    lbl  = f'~{mins} min' if secs == 0 else f'~{mins} min {secs}s'
    ax.text(total + 10, y_video[xi], f'{lbl} ({total}s)',
            va='center', fontsize=8, color='#222222', fontweight='bold')

# ── Draw podcast bars ────────────────────────────────────────────────────────
cumulative_p = np.zeros(n_podcast)
for stage in reversed(PODCAST_STAGES):
    vals = np.array(PODCAST_DATA[stage], dtype=float)
    ax.barh(y_podcast, vals, left=cumulative_p, height=bar_h,
            color=PODCAST_COLORS[stage], edgecolor='white', linewidth=0.6, zorder=3)
    for xi, (v, b) in enumerate(zip(vals, cumulative_p)):
        if v >= 14:
            ax.text(b + v / 2, y_podcast[xi], f'{v:.0f}s',
                    ha='center', va='center', fontsize=7, color='white', fontweight='bold')
    cumulative_p += vals

for xi, total in enumerate(PODCAST_TOTALS_S):
    mins = int(total) // 60
    secs = total % 60
    ax.text(total + 10, y_podcast[xi], f'~{mins} min {secs:.0f}s ({total:.0f}s)',
            va='center', fontsize=8, color='#222222', fontweight='bold')

# ── Separator line between video and podcast groups ──────────────────────────
sep_y = (y_video[-1] + y_podcast[0]) / 2
ax.axhline(sep_y, color='#BBBBBB', linewidth=1.0, linestyle='--', zorder=1)

# Section labels — placed just above the first bar of each group
ax.annotate('VIDEO PIPELINE', xy=(0, y_video[-1] + 0.55),
            fontsize=8, color='#2563EB', fontweight='bold',
            fontstyle='normal', va='bottom')
ax.annotate('PODCAST PIPELINE', xy=(0, y_podcast[-1] + 0.55),
            fontsize=8, color='#059669', fontweight='bold',
            fontstyle='normal', va='bottom')

# ── Axes ─────────────────────────────────────────────────────────────────────
ax.set_yticks(all_y)
ax.set_yticklabels(all_labels, fontsize=10.5)
ax.set_xlabel('Processing Time (seconds)', fontsize=10, labelpad=8)
ax.set_xlim(0, 850)
ax.xaxis.set_major_locator(ticker.MultipleLocator(60))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(30))

ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ax2.set_xlabel('Processing Time (minutes)', fontsize=9, labelpad=6)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(60))
ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{v/60:.0f}' if v >= 0 else ''))
ax2.spines['top'].set_color('#CCCCCC')

ax.set_title(
    'Per-Stage Processing Time: Video & Podcast Pipelines\n(15 Document Pages, 2 Speakers for Podcast)',
    fontsize=11.5, fontweight='bold', pad=16,
)

# ── Legend ───────────────────────────────────────────────────────────────────
video_patches = [
    mpatches.Patch(color=VIDEO_COLORS[s], label=s)
    for s in reversed(VIDEO_STAGES)
]
podcast_patches = [
    mpatches.Patch(color=PODCAST_COLORS[s], label=s)
    for s in reversed(PODCAST_STAGES)
    if s not in ('PDF Parsing',)   # skip duplicate
]

from matplotlib.lines import Line2D
sep_handle = Line2D([0], [0], color='none', label='── Podcast stages ──')

legend = ax.legend(
    handles=video_patches + [sep_handle] + podcast_patches,
    loc='lower right',
    fontsize=7.5,
    framealpha=0.93,
    edgecolor='#cccccc',
    title='Pipeline Stage',
    title_fontsize=8,
    ncol=1,
    bbox_to_anchor=(1.0, 0.01),
)


fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig('graph2_processing_time.pdf', dpi=300, bbox_inches='tight')
fig.savefig('graph2_processing_time.png', dpi=300, bbox_inches='tight')
print('Saved → graph2_processing_time.pdf / .png')


Saved → graph2_processing_time.pdf / .png
